In [ ]:
"""01_data_exploration.ipynb — Preliminary exploration of ClinVar missense variants."""
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from load_clinvar import load_and_label_clinvar, print_funnel

DATA_DIR = Path.cwd().parent / "data"
RAW_FILE = DATA_DIR / "raw" / "variant_summary.txt.gz"
INTERIM_OUT = DATA_DIR / "interim" / "labeled_snvs.parquet"

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

In [ ]:
# %%
df_labeled, funnel = load_and_label_clinvar(RAW_FILE, return_funnel=True)
print_funnel(funnel)

In [ ]:
# %%
completeness = df_labeled.notna().mean().sort_values()
completeness.to_frame("non_null_fraction")

In [ ]:
# The file is tab-delimited and gzipped. compression="infer" reads .gz directly.
# low_memory=False avoids pandas' mixed-dtype warnings on this wide file.
df = pd.read_csv(
    CLINVAR_FILE,
    sep="\t",
    compression="infer",
    low_memory=False,
    dtype=str,            # read everything as string first; coerce deliberately later
)

print(f"Loaded {len(df):,} rows × {df.shape[1]} columns")
df.head()

In [ ]:
# What columns do we have, and how complete is each?
print("Columns:\n", list(df.columns), "\n")

# Completeness overview — fraction of non-null values per column.
completeness = df.notna().mean().sort_values()
completeness.to_frame("non_null_fraction")

In [ ]:
# 1. One genome assembly only — ClinVar lists the same variant under both
#    GRCh37 and GRCh38, so not filtering would double-count.
df_38 = df[df["Assembly"] == "GRCh38"].copy()

# 2. Single nucleotide variants only (a missense change is a single base swap).
df_snv = df_38[df_38["Type"] == "single nucleotide variant"].copy()

print(f"GRCh38 rows:        {len(df_38):,}")
print(f"  of which SNVs:    {len(df_snv):,}")

In [ ]:
# See the full landscape of label values before deciding what to keep.
df_snv["ClinicalSignificance"].value_counts().head(20)

In [ ]:
# Standard ClinVar label-cleaning: keep only confident pathogenic / benign calls.
# "Likely" calls are conventionally folded in; uncertain/conflicting are dropped.
PATHOGENIC = {"Pathogenic", "Likely pathogenic", "Pathogenic/Likely pathogenic"}
BENIGN     = {"Benign", "Likely benign", "Benign/Likely benign"}

def to_label(sig: str):
    if sig in PATHOGENIC:
        return 1
    if sig in BENIGN:
        return 0
    return np.nan   # uncertain, conflicting, drug-response, etc. → excluded

df_snv["label"] = df_snv["ClinicalSignificance"].map(to_label)

# Keep only rows that resolved to a clean binary label.
df_labeled = df_snv.dropna(subset=["label"]).copy()
df_labeled["label"] = df_labeled["label"].astype(int)

print(f"Confidently labeled SNVs: {len(df_labeled):,}")

In [ ]:
counts = df_labeled["label"].value_counts().rename({0: "benign", 1: "pathogenic"})
print(counts)
print(f"\nPathogenic fraction: {df_labeled['label'].mean():.1%}")

In [ ]:
# The HGVS Name looks like:
#   NM_000546.6(TP53):c.524G>A (p.Arg175His)
# Your feature extractor needs the p.(...) protein-level part.
has_protein = df_labeled["Name"].str.contains(r"p\.", regex=True, na=False)
print(f"Labeled SNVs with a protein-level (p.) consequence: {has_protein.sum():,} "
      f"({has_protein.mean():.1%})")

# Peek at a few real Names to confirm the format matches what features.py expects.
df_labeled.loc[has_protein, ["GeneSymbol", "Name", "ClinicalSignificance", "label"]].head()

In [ ]:
df_labeled.loc[has_protein, "GeneSymbol"].value_counts().head(15)

In [ ]:
# ===== CELL 2: classify protein consequences, isolate missense ===============
# Adds p_change / consequence / p_hgvs and prints the data funnel.
df_pc = add_protein_change_columns(df_labeled, name_col="Name")
df_missense = df_pc[df_pc["consequence"] == "missense"].copy()
print(f"\nMissense rows: {len(df_missense):,}")
print(df_missense["label"].value_counts().rename({0: "benign", 1: "pathogenic"}))

In [ ]:
# ===== CELL 3: build the feature matrix (skip-and-count) =====================
# We extract the SEQUENCE-FREE features here: 8 of the 10 features in
# features.py need no protein sequence. That means we can build the whole
# matrix WITHOUT fetching a single UniProt sequence — no isoform-mismatch
# data loss, no network. (rel_position / seq_length are deliberately deferred;
# add them later only if you fetch sequences and have time.)
 
rows = []
skipped = {"parse_error": 0}
for _, r in df_missense.iterrows():
    try:
        feats = extract_features(r["p_hgvs"])   # no sequence -> 8 features
        feats["label"] = int(r["label"])
        feats["gene"] = r["GeneSymbol"]
        rows.append(feats)
    except Exception:
        skipped["parse_error"] += 1
 
X_df = pd.DataFrame(rows)
print(f"Built matrix: {X_df.shape[0]:,} variants × "
      f"{X_df.shape[1] - 2} features (skipped {skipped['parse_error']:,})")
 
feature_cols = [c for c in X_df.columns if c not in ("label", "gene")]
print("Features:", feature_cols)
X_df.head()

In [ ]:
# ===== CELL 4: a quick look at class balance & feature/label relationship ====
print(X_df["label"].value_counts(normalize=True).rename({0: "benign", 1: "pathogenic"}))
 
# Mean of each feature by class — a fast sanity check that features carry signal.
# Expect pathogenic variants to have MORE negative blosum62, LARGER abs deltas.
X_df.groupby("label")[feature_cols].mean().T

In [ ]:
# ===== CELL 5: train / test split (stratified) ==============================
X = X_df[feature_cols].values
y = X_df["label"].values
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f"Train: {len(y_train):,}  Test: {len(y_test):,}  "
      f"(test pathogenic frac: {y_test.mean():.1%})")

In [ ]:
# ===== CELL 6: Model A — Logistic Regression (interpretable baseline) ========
# Scale + LR. class_weight balanced because ClinVar is usually pathogenic-heavy.
logreg = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, class_weight="balanced",
                       random_state=RANDOM_STATE),
)
logreg.fit(X_train, y_train)
p_lr = logreg.predict_proba(X_test)[:, 1]
 
print("LOGISTIC REGRESSION")
print(f"  ROC-AUC: {roc_auc_score(y_test, p_lr):.3f}")
print(f"  PR-AUC : {average_precision_score(y_test, p_lr):.3f}")
 
# Coefficients = interpretable feature effects (in standardized units).
coefs = pd.Series(
    logreg.named_steps["logisticregression"].coef_[0], index=feature_cols
).sort_values()
print("\nLR coefficients (standardized; + => pushes toward pathogenic):")
print(coefs.to_string())

In [ ]:
# ===== CELL 7: Model B — Gradient Boosting (the workhorse) ===================
hgb = HistGradientBoostingClassifier(
    max_iter=300, learning_rate=0.05, max_depth=4,
    l2_regularization=1.0, random_state=RANDOM_STATE,
)
hgb.fit(X_train, y_train)
p_hgb = hgb.predict_proba(X_test)[:, 1]
 
print("HIST GRADIENT BOOSTING")
print(f"  ROC-AUC: {roc_auc_score(y_test, p_hgb):.3f}")
print(f"  PR-AUC : {average_precision_score(y_test, p_hgb):.3f}")

In [ ]:
# ===== CELL 8: 5-fold stratified CV (don't trust a single split) ============
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
for name, model in [("LogReg", logreg), ("HistGB", hgb)]:
    scores = cross_val_score(model, X, y, cv=cv, scoring="roc_auc")
    print(f"{name}: ROC-AUC {scores.mean():.3f} ± {scores.std():.3f}  {np.round(scores,3)}")

In [ ]:
# ===== CELL 9: confusion matrix + classification report (use better model) ==
best_p = p_hgb if roc_auc_score(y_test, p_hgb) >= roc_auc_score(y_test, p_lr) else p_lr
best_name = "HistGB" if best_p is p_hgb else "LogReg"
y_pred = (best_p >= 0.5).astype(int)
 
print(f"Best model: {best_name}")
print("\nConfusion matrix [rows=true, cols=pred] (0=benign,1=pathogenic):")
print(confusion_matrix(y_test, y_pred))
print("\n", classification_report(y_test, y_pred, target_names=["benign", "pathogenic"]))

In [ ]:
# ===== CELL 10: ROC + PR curves =============================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
 
for name, p in [("LogReg", p_lr), ("HistGB", p_hgb)]:
    fpr, tpr, _ = roc_curve(y_test, p)
    ax1.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test,p):.3f})")
ax1.plot([0, 1], [0, 1], "k--", lw=0.8)
ax1.set(xlabel="False Positive Rate", ylabel="True Positive Rate", title="ROC")
ax1.legend()
 
for name, p in [("LogReg", p_lr), ("HistGB", p_hgb)]:
    prec, rec, _ = precision_recall_curve(y_test, p)
    ax2.plot(rec, prec, label=f"{name} (AP={average_precision_score(y_test,p):.3f})")
ax2.set(xlabel="Recall", ylabel="Precision", title="Precision-Recall")
ax2.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ===== CELL 11: permutation importance (model-agnostic, honest) =============
from sklearn.inspection import permutation_importance
r = permutation_importance(hgb, X_test, y_test, n_repeats=10,
                           random_state=RANDOM_STATE, scoring="roc_auc")
imp = pd.Series(r.importances_mean, index=feature_cols).sort_values(ascending=False)
print("Permutation importance (drop in ROC-AUC when feature shuffled):")
print(imp.to_string())

In [ ]:
# ===== CELL 12: GROUPED sanity check — beware gene leakage ==================
# A subtle ML-in-biology trap: if the SAME gene appears in train and test, the
# model can memorize "variants in gene X tend to be pathogenic" rather than
# learning variant-level chemistry. A tougher, more honest evaluation holds
# entire genes out. Compare this AUC to CELL 8 — a big drop reveals leakage.
from sklearn.model_selection import GroupShuffleSplit
groups = X_df["gene"].values
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
tr_idx, te_idx = next(gss.split(X, y, groups))
hgb_g = HistGradientBoostingClassifier(
    max_iter=300, learning_rate=0.05, max_depth=4,
    l2_regularization=1.0, random_state=RANDOM_STATE,
).fit(X[tr_idx], y[tr_idx])
p_g = hgb_g.predict_proba(X[te_idx])[:, 1]
print(f"Gene-held-out ROC-AUC: {roc_auc_score(y[te_idx], p_g):.3f}  "
      f"(compare to the random-split CV above)")


In [ ]:
# ===== CELL 13: persist the model for the FastAPI service ===================
import joblib
joblib.dump(
    {"model": hgb, "feature_cols": feature_cols},
    PROJECT_ROOT / "model.joblib",
)
print("Saved model.joblib")